# box-array-to-tensor-with-recipe — worked example 3: Box a binary op carrying a kwarg, with mixed parents

> Worked example from [Delta Drills](https://delta-drills.vercel.app). Atom: `box-array-to-tensor-with-recipe`.

**This is a worked example — read it, run it, follow the reasoning.** It is study material, not a graded drill (no completion beacon). When the steps feel obvious, move to the faded version, then the full drill.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)
# === manual autograd primitives — shared across all drills in this folder ===
from dataclasses import dataclass, field
from typing import Any, Callable, Optional

grad_tracking_enabled = True

@dataclass
class Recipe:
    func: Optional[Callable] = None
    args: tuple = ()
    kwargs: dict = field(default_factory=dict)
    parents: dict = field(default_factory=dict)

class MiniTensor:
    """Minimal Tensor wrapper for the ARENA-style manual-autograd drills.
    Wraps a raw `torch.Tensor` in `.array`. Carries optional `.recipe`,
    `.requires_grad`, and `.grad` (the accumulated gradient at leaves)."""
    def __init__(self, array, requires_grad: bool = False, recipe=None):
        self.array = array
        self.requires_grad = requires_grad
        self.recipe = recipe
        self.grad = None
    def __repr__(self):
        return f'MiniTensor({self.array!r}, requires_grad={self.requires_grad})'

## Concept

Real wrappers see a mix of `MiniTensor` and raw scalar args plus keyword arguments. The boxing half must (1) unbox every positional arg, (2) carry the original `kwargs` into the Recipe verbatim, and (3) build a `parents` dict keyed by the original argnum holding only the rg=True MiniTensors. The Recipe always stores the 4-tuple `(func, raw_args, kwargs, parents)` so backward can replay and route.

## Worked solution

We wrap `axpy(a, b, scale=...)` = `scale*a + b` with two MiniTensor inputs (only the first requires grad) and a scalar kwarg.

1. **Unbox positionals.** `raw_args = (a.array, b.array)`. Both MiniTensors give up their `.array`.
2. **Run forward with kwargs.** `out_raw = axpy(*raw_args, **kwargs)` where `kwargs={'scale': 2.0}`.
3. **Gate.** `any_rg` is True because `a.requires_grad` is True (b is False), so `requires_grad = True`.
4. **Build parents by argnum.** Walk the original `args` with `enumerate`; keep only rg=True MiniTensors. `a` is at index 0 and qualifies; `b` at index 1 is rg=False and is skipped. Result: `{0: a}`. The index is the *original* slot, not renumbered.
5. **Box + attach Recipe.** `out = MiniTensor(out_raw, requires_grad=True)` then `out.recipe = Recipe(axpy, raw_args, kwargs, {0: a})`. The kwargs flow into the Recipe unchanged so backward sees the same `scale`.

In [ ]:
from typing import Callable
from dataclasses import dataclass

class MiniTensor:
    def __init__(self, array, requires_grad=False):
        self.array = np.asarray(array)
        self.requires_grad = requires_grad
        self.recipe = None

@dataclass
class Recipe:
    func: Callable
    args: tuple
    kwargs: dict
    parents: dict

def axpy(a, b, scale=1.0):
    return scale * a + b

def box_axpy(args, kwargs):
    raw_args = tuple(a.array if isinstance(a, MiniTensor) else a for a in args)
    any_rg = any(isinstance(a, MiniTensor) and a.requires_grad for a in args)
    requires_grad = any_rg
    out_raw = axpy(*raw_args, **kwargs)
    parents = {}
    if requires_grad:
        parents = {idx: a for idx, a in enumerate(args)
                   if isinstance(a, MiniTensor) and a.requires_grad}
    out = MiniTensor(out_raw, requires_grad=requires_grad)
    if requires_grad:
        out.recipe = Recipe(axpy, raw_args, kwargs, parents)
    return out

a = MiniTensor(np.array([1.0, 2.0]), requires_grad=True)
b = MiniTensor(np.array([10.0, 20.0]), requires_grad=False)
out = box_axpy((a, b), {"scale": 2.0})
print("array        :", out.array)
print("requires_grad:", out.requires_grad)
print("recipe kwargs:", out.recipe.kwargs)
print("parents keys :", list(out.recipe.parents.keys()))
print("parent[0] is a:", out.recipe.parents[0] is a)